# 🚀 Path 2: Image-to-Signal
**Model**: Digitization Regressor

### Configuration:
- **Dataset**: MIT-BIH Arrhythmia (10% Sample for Testing)
- **Epochs**: 2
- **Environment**: Production-Ready Torch Pipeline

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import os
from torch.utils.data import DataLoader, TensorDataset, Dataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_PATH = '../../data/mitbih_train.csv'

def load_production_data(sample_frac=0.1):
    if not os.path.exists(DATA_PATH): return None
    df = pd.read_csv(DATA_PATH, header=None)
    return df.sample(frac=sample_frac, random_state=42)

df_sample = load_production_data(0.1)
print(f'📦 Loaded {len(df_sample)} rows (10% of dataset)')

In [ ]:
import cv2
import matplotlib.pyplot as plt

def fast_signal_to_image(signal, img_size=(128, 128)):
    # Faster generation using direct numpy plotting for production pipelines
    img = np.zeros((img_size[0], img_size[1]), dtype=np.float32)
    x_coords = np.linspace(0, img_size[1]-1, len(signal)).astype(int)
    y_coords = ((1 - signal) * (img_size[0]-1)).astype(int)
    for i in range(len(x_coords)-1):
        cv2.line(img, (x_coords[i], y_coords[i]), (x_coords[i+1], y_coords[i+1]), 1.0, 1)
    return img

class DigitizerCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), 
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 512),
            nn.ReLU(),
            nn.Linear(512, 187)
        )
    def forward(self, x): return self.regressor(self.features(x))

model = DigitizerCNN().to(DEVICE)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print('🎨 Generating training images on-the-fly...')
X_imgs = torch.tensor([fast_signal_to_image(s) for s in df_sample.iloc[:200, :187].values], dtype=torch.float32).unsqueeze(1).to(DEVICE)
y_sigs = torch.tensor(df_sample.iloc[:200, :187].values, dtype=torch.float32).to(DEVICE)

print('🚀 Production Training: Image-to-Signal Reconstruction')
for epoch in range(2):
    optimizer.zero_grad()
    preds = model(X_imgs)
    loss = criterion(preds, y_sigs)
    loss.backward()
    optimizer.step()
    print(f'Epoch {epoch+1}: Reconstruction Loss={loss.item():.6f}')

if not os.path.exists('models'): os.makedirs('models')
torch.save(model.state_dict(), 'models/image_to_signal.pth')
print('✅ Model Saved.')